# 02 — Pipeline & Inferenz

Phase 3 (Pipeline bauen + erste Inferenz auf 12 Hand-Gold-Anzeigen), Phase 4 (Iterationen A + B — pro Iteration eigener Predictions-Dateiname und Run-Header-Update), Phase 6 (voller Korpus auf 7B + 3B-Kontrast auf euler).

Cheatsheets: `CHEATSHEETS/transformers-konzepte.md` (Modell, Chat-Template, JSON-Parsing), `CHEATSHEETS/gpu-zugang.md` (Spawn, GPU-Wahl, Memory).

## Run-Header

| Feld | Wert |
|---|---|
| Datum | _2026-06-01_ |
| Modell | _ `Qwen/Qwen2.5-7B-Instruct` |
| Server | _ euler |
| GPU-Index | _ [0, 1, 2, 3] |
| Schema-Datei | `SCHEMA.md` |
| Aktueller Run-Tag | _ `Iteration B` |
| Predictions-Datei | _ `predictions_iterB.jsonl` |
| Truncation | _ 2000 Zeichen |

Bei jeder neuen Iteration: Run-Tag + Predictions-Datei + Datum aktualisieren.

## Phase 3 — Pipeline bauen + Baseline-Inferenz

In [1]:
# ------------------------------------------------------------
# GPU-Auswahl
# ------------------------------------------------------------
# Erlaubt: eine oder mehrere Nummern zwischen 0 und 3
# Beispiele:
# GPU_IDS = [0]              # nur physische GPU 0 nutzen
# GPU_IDS = [1]              # nur physische GPU 1 nutzen
# GPU_IDS = [0, 1]           # Modellgewichte auf GPU 0 und 1 verteilen
# GPU_IDS = [0, 1, 2, 3]     # Modellgewichte auf alle 4 GPUs verteilen

GPU_IDS = [0, 1, 2, 3]

assert isinstance(GPU_IDS, list) and len(GPU_IDS) >= 1, "GPU_IDS muss eine nicht-leere Liste sein, z. B. [0] oder [0, 1]."
assert all(isinstance(g, int) for g in GPU_IDS), "Alle GPU_IDS müssen ganze Zahlen sein."
assert all(0 <= g <= 3 for g in GPU_IDS), "Erlaubt sind nur GPU-Nummern zwischen 0 und 3."
assert len(GPU_IDS) == len(set(GPU_IDS)), "GPU_IDS darf keine Duplikate enthalten."

# WICHTIG:
# CUDA_VISIBLE_DEVICES muss gesetzt werden, bevor torch/CUDA aktiv genutzt wird.
# Wenn du GPU_IDS änderst: Kernel neu starten und Notebook von oben ausführen.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(str(g) for g in GPU_IDS)

# Kann Speicherfragmentierung reduzieren.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

import json
import re
from pathlib import Path

import pandas as pd
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

print("Gewählte physische GPUs:", GPU_IDS)
print("CUDA verfügbar:", torch.cuda.is_available())
print("Für PyTorch sichtbare GPUs:", torch.cuda.device_count())

if torch.cuda.is_available():
    for local_idx in range(torch.cuda.device_count()):
        physical_idx = GPU_IDS[local_idx]
        free, total = torch.cuda.mem_get_info(local_idx)
        print(
            f"lokal cuda:{local_idx} -> physisch GPU {physical_idx}: "
            f"{torch.cuda.get_device_name(local_idx)} | frei: {free/1024**3:.1f} GB / {total/1024**3:.1f} GB"
        )
else:
    print("WARNUNG: CUDA ist nicht verfügbar. Das Notebook würde auf CPU laufen und sehr langsam sein.")


<jemalloc>: Unsupported system page size


Gewählte physische GPUs: [0, 1, 2, 3]
CUDA verfügbar: True
Für PyTorch sichtbare GPUs: 4
lokal cuda:0 -> physisch GPU 0: Tesla V100-SXM2-16GB | frei: 15.1 GB / 15.8 GB
lokal cuda:1 -> physisch GPU 1: Tesla V100-SXM2-16GB | frei: 15.1 GB / 15.8 GB
lokal cuda:2 -> physisch GPU 2: Tesla V100-SXM2-16GB | frei: 15.1 GB / 15.8 GB
lokal cuda:3 -> physisch GPU 3: Tesla V100-SXM2-16GB | frei: 15.1 GB / 15.8 GB


In [2]:
GOLD_PATH = Path("../annotation/meine_gold.csv")
DATA_PATH = Path("../daten/annotations_auswahl.csv")
PRED_PATH = Path("../daten/predictions_iterB.jsonl")

# Falls du das Notebook testweise direkt neben den Dateien ausführst:
if not GOLD_PATH.exists() and Path("meine_gold.csv").exists():
    GOLD_PATH = Path("meine_gold.csv")
if not DATA_PATH.exists() and Path("annotations_auswahl.csv").exists():
    DATA_PATH = Path("annotations_auswahl.csv")

GOLD_PATH, DATA_PATH, PRED_PATH


(PosixPath('../annotation/meine_gold.csv'),
 PosixPath('../daten/annotations_auswahl.csv'),
 PosixPath('../daten/predictions_iterB.jsonl'))

In [3]:
gold = pd.read_csv(GOLD_PATH)
anzeigen = pd.read_csv(DATA_PATH)

if "id" in gold.columns and "refnr" not in gold.columns:
    gold = gold.rename(columns={"id": "refnr"})

gold["refnr"] = gold["refnr"].astype(str).str.strip()
anzeigen["refnr"] = anzeigen["refnr"].astype(str).str.strip()

print("Gold:", gold.shape)
print("Anzeigen:", anzeigen.shape)

gold.head()


Gold: (12, 8)
Anzeigen: (12, 11)


,refnr,homeoffice,vertragsart,erfahrungslevel,gehalt_min_eur,gehalt_zeitraum,skills_top3,notiz
0,15939-BB-633097-7878-9999-S,ja,festanstellung,junior,NaN,NaN,FMECA|Reliability Block Diagrams|ILS,"Junior entschieden ohne expliziete Nennung, du..."
1,15939-BB-633095-7878-7490-S,ja,festanstellung,senior,NaN,NaN,OPUS Suite|Python,NaN
2,15939-BB-633455-7878-6343-S,nicht_genannt,festanstellung,senior,NaN,NaN,Data-Lake-/Lakehouse-Architektur|Data-Ingestio...,"festanstellung anhand 30 Tage Urlaub, senior a..."
3,15939-BB-633457-7878-2175-S,nicht_genannt,festanstellung,senior,NaN,NaN,MLOps|Platform Engineering|Model Deployment,festanstellung anhand von 30 Tage Urlaub
4,18777-931781141-S,nicht_genannt,festanstellung,nicht_genannt,NaN,NaN,Siemens Polarion|IBM Rational DOORS|systems en...,unbefristet -> festanstellung


In [4]:
# Nur die 12 Anzeigen verwenden, die im Gold enthalten sind
subset = anzeigen[anzeigen["refnr"].isin(gold["refnr"])].copy()

print("Anzahl ausgewählte Anzeigen:", len(subset))

# Sicherheitscheck: Fehlen Anzeigen aus dem Gold?
missing = set(gold["refnr"]) - set(subset["refnr"])
if missing:
    print("FEHLENDE REFNRs:", missing)
else:
    print("Alle Gold-Anzeigen wurden gefunden.")

subset[["refnr", "titel", "firma", "text"]].head()

Anzahl ausgewählte Anzeigen: 12
Alle Gold-Anzeigen wurden gefunden.


,refnr,titel,firma,text
0,15939-BB-633097-7878-9999-S,Data Analyst im Marineschiffbau (m/w/d),Rheinmetall AG,Moderne Marineschiffe sind hochkomplexe System...
1,15939-BB-633095-7878-7490-S,(Senior) Data Analyst - Lifecycle-Analysen im ...,Rheinmetall AG,Marineschiffe sind hochkomplexe Systeme mit Le...
2,15939-BB-633455-7878-6343-S,Data Engineer (m/w/d),Rheinmetall AG,Rheinmetall Digital GmbH\n\nWHAT WE ARE LOOKIN...
3,15939-BB-633457-7878-2175-S,MLOps Engineer (m/w/d),Rheinmetall AG,Rheinmetall Digital GmbH\n\nWOFÜR WIR SIE SUCH...
4,18777-931781141-S,Requirements Engineer (m/w/d),OHB-System AG,#### Your Tasks\n\n- Being the project’s focal...


In [5]:
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if torch.cuda.is_available():
    if torch.cuda.device_count() == 1:
        # Eine GPU: klassisch laden und auf diese GPU schieben.
        # Achtung: Qwen 7B in float32 kann auf 16 GB V100 trotzdem zu groß sein.
        print("Lade Modell auf eine GPU ...")
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float32,
        ).to("cuda:0")
    else:
        # Mehrere GPUs: Modellgewichte über GPUs verteilen.
        # Das ist etwas anderes als mehrere komplette Modellkopien zu laden.
        print("Lade Modell verteilt über mehrere GPUs ...")

        max_memory = {
            local_idx: "14GiB" for local_idx in range(torch.cuda.device_count())
        }

        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float32,
            device_map="auto",
            max_memory=max_memory,
            low_cpu_mem_usage=True,
        )
else:
    print("Lade Modell auf CPU. Das ist nur für Debugging sinnvoll und sehr langsam.")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float32,
    )

model.eval()

# Device, auf das die Input-Tensoren gelegt werden.
# Bei geshardetem Modell liegt die Embedding-Schicht meist auf cuda:0.
INPUT_DEVICE = next(model.parameters()).device

print("Modell geladen:", MODEL_NAME)
print("Input-Device:", INPUT_DEVICE)

if hasattr(model, "hf_device_map"):
    print("Device Map:")
    for name, device in model.hf_device_map.items():
        print(f"  {name}: {device}")


/opt/conda/envs/torch/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Modell geladen: Qwen/Qwen2.5-7B-Instruct
Input-Device: cuda:0
Device Map:
  model.embed_tokens: 0
  model.layers.0: 0
  model.layers.1: 0
  model.layers.2: 0
  model.layers.3: 0
  model.layers.4: 1
  model.layers.5: 1
  model.layers.6: 1
  model.layers.7: 1
  model.layers.8: 1
  model.layers.9: 1
  model.layers.10: 1
  model.layers.11: 1
  model.layers.12: 1
  model.layers.13: 2
  model.layers.14: 2
  model.layers.15: 2
  model.layers.16: 2
  model.layers.17: 2
  model.layers.18: 2
  model.layers.19: 2
  model.layers.20: 2
  model.layers.21: 2
  model.layers.22: 3
  model.layers.23: 3
  model.layers.24: 3
  model.layers.25: 3
  model.layers.26: 3
  model.layers.27: 3
  model.norm: 3
  lm_head: 3


In [6]:
SYSTEM_PROMPT = """
Du bist ein Information-Extraction-System für deutsche Stellenanzeigen.

Extrahiere genau diese Felder aus der Stellenanzeige.
Gib ausschließlich valides JSON zurück. Keine Erklärung, keine Markdown-Codeblöcke.

Erlaubtes JSON-Schema:

{
  "homeoffice": "ja|teilweise|nein|remote|nicht_genannt",
  "vertragsart": "ausbildung|festanstellung|praktikum|werkstudent|sonstiges",
  "erfahrungslevel": "junior|mid|senior|egal|nicht_genannt",
  "gehalt_min_eur": 50000 oder null,
  "gehalt_zeitraum": "monat|jahr|null",
  "skills_top3": ["Skill 1", "Skill 2", "Skill 3"]
}

Regeln:
- homeoffice:
  - "remote" = 100% Homeoffice / deutschlandweit von zuhause
  - "teilweise" = hybrid, mobiles Arbeiten, anteilig Homeoffice
  - "ja" = Homeoffice erwähnt, aber Modus unklar
  - "nein" = explizit kein Homeoffice / Präsenzpflicht
  - "nicht_genannt" = keine Aussage dazu
- vertragsart:
  - Ausbildung = ausbildung
  - reguläre Stelle = festanstellung
  - Praktikum = praktikum
  - Werkstudent = werkstudent
  - sonstiges nur wenn nichts davon passt
- erfahrungslevel:
  - Ausbildung zählt immer als junior
  - Senior im Titel oder viele Jahre Erfahrung = senior
  - Erfahrung erforderlich ohne Senior = mid
  - Berufseinsteiger / bis 2 Jahre = junior
  - Junior bis Senior willkommen = egal
  - Wähle "senior" nur, wenn im Titel oder Text explizit "Senior" steht oder eine hohe Berufserfahrung genannt wird, z. B. "mehrjährige Erfahrung", "5 Jahre", "8+ Jahre", "Experte".
  - Wähle "mid" nur, wenn konkrete Berufserfahrung verlangt wird, aber keine Senior-Rolle erkennbar ist.
  - Wähle "junior" nur bei Ausbildung, Berufseinstieg, Trainee, Praktikum oder ausdrücklich geringer/erster Erfahrung.
  - Wähle "egal" nur, wenn ausdrücklich mehrere Level akzeptiert werden, z. B. "Junior bis Senior willkommen".
  - Wähle "nicht_genannt", wenn keine konkrete Aussage zum Erfahrungslevel gemacht wird.
  - Leite das Erfahrungslevel nicht nur aus der Komplexität der Aufgaben ab.
- gehalt_min_eur:
  - Nur konkrete Zahlen extrahieren.
  - Bei "50.000 bis 60.000 Euro" nimm 50000.
  - Bei "nach Vereinbarung" oder "attraktive Vergütung" nimm null.
- gehalt_zeitraum:
  - Wenn gehalt_min_eur null ist, dann gehalt_zeitraum auch null.
  - Monatsgehalt = monat
  - Jahresgehalt = jahr
- skills_top3:
  - Maximal 3 technische Skills oder Tools.
  - Keine Soft Skills.
  - Keine Sprachen wie Deutsch/Englisch.
  - Keine Schulabschlüsse.
  - Wenn keine technischen Skills genannt sind: []
  - Extrahiere maximal 3 konkrete technische Skills, Tools, Methoden, Plattformen, Frameworks oder Programmiersprachen.
- Bevorzuge spezifische Begriffe aus dem Text, z. B. "Python", "OPUS Suite", "FMECA", "IBM Rational DOORS", "MLOps".
- Vermeide generische Tätigkeiten wie "Datenanalyse", "Auswertung", "Recherche", "Reporting", wenn spezifischere technische Skills im Text vorhanden sind.
- Keine Soft Skills, keine Sprachen, keine allgemeinen Studienfächer.
- Gib die Skills als JSON-Liste aus.

Regeln für erfahrungslevel:


Regeln für skills_top3:


Beispiel-Ausgabe:
{
  "homeoffice": "teilweise",
  "vertragsart": "festanstellung",
  "erfahrungslevel": "mid",
  "gehalt_min_eur": 50000,
  "gehalt_zeitraum": "jahr",
  "skills_top3": ["Python", "SQL", "Power BI"]
}
"""

def build_messages(job_text, max_chars=2000):
    text = str(job_text)[:max_chars]

    user_prompt = f"""
Stellenanzeige:
{text}

Extrahiere die Felder als valides JSON.
"""

    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt}
    ]


In [7]:
def generate_response(messages, max_new_tokens=300):
    """
    Generiert eine Modellantwort.

    Funktioniert sowohl mit einem normalen Modell auf einer GPU
    als auch mit einem per device_map geshardeten Modell auf mehreren GPUs.
    """
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True
    ).to(INPUT_DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Nur die neu generierten Tokens decodieren, nicht den ganzen Prompt.
    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()


In [8]:
EXPECTED_FIELDS = [
    "homeoffice",
    "vertragsart",
    "erfahrungslevel",
    "gehalt_min_eur",
    "gehalt_zeitraum",
    "skills_top3"
]

def extract_json(output):
    match = re.search(r"\{.*\}", output, re.DOTALL)
    if not match:
        return None

    try:
        parsed = json.loads(match.group())
    except json.JSONDecodeError:
        return None

    return parsed


def normalize_prediction(parsed):
    """
    Sorgt dafür, dass alle erwarteten Felder vorhanden sind.
    Das ersetzt keine inhaltliche Validierung, verhindert aber kaputte Zeilen.
    """
    if parsed is None:
        return None

    normalized = {}
    for field in EXPECTED_FIELDS:
        normalized[field] = parsed.get(field, None)

    # skills_top3 absichern
    if normalized["skills_top3"] is None:
        normalized["skills_top3"] = []
    elif isinstance(normalized["skills_top3"], str):
        normalized["skills_top3"] = [s.strip() for s in normalized["skills_top3"].split("|") if s.strip()]
    elif not isinstance(normalized["skills_top3"], list):
        normalized["skills_top3"] = []

    normalized["skills_top3"] = normalized["skills_top3"][:3]

    # Konsistenzregel Gehalt
    if pd.isna(normalized["gehalt_min_eur"]) or normalized["gehalt_min_eur"] in ["", "null", "None"]:
        normalized["gehalt_min_eur"] = None
        normalized["gehalt_zeitraum"] = None

    return normalized


In [9]:
test_job = subset.iloc[0]

messages = build_messages(test_job["text"])
raw_output = generate_response(messages)

print("REFNR:", test_job["refnr"])
print("ROHE MODELLANTWORT:")
print(raw_output)

parsed = extract_json(raw_output)
normalized = normalize_prediction(parsed)

print("\nNORMALISIERT:")
normalized


/opt/conda/envs/torch/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:509: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


REFNR: 15939-BB-633097-7878-9999-S
ROHE MODELLANTWORT:
{
  "homeoffice": "nicht_genannt",
  "vertragsart": "festanstellung",
  "erfahrungslevel": "mid",
  "gehalt_min_eur": null,
  "gehalt_zeitraum": null,
  "skills_top3": ["Python", "FMECA", "OPUS Suite"]
}

NORMALISIERT:


{'homeoffice': 'nicht_genannt',
 'vertragsart': 'festanstellung',
 'erfahrungslevel': 'mid',
 'gehalt_min_eur': None,
 'gehalt_zeitraum': None,
 'skills_top3': ['Python', 'FMECA', 'OPUS Suite']}

In [10]:
def predict_row(row):
    messages = build_messages(row["text"])
    raw_output = generate_response(messages)

    parsed = extract_json(raw_output)
    normalized = normalize_prediction(parsed)

    if normalized is None:
        prediction = {
            "refnr": row["refnr"],
            "parse_ok": False,
            "raw_output": raw_output,
            "homeoffice": None,
            "vertragsart": None,
            "erfahrungslevel": None,
            "gehalt_min_eur": None,
            "gehalt_zeitraum": None,
            "skills_top3": []
        }
    else:
        prediction = {
            "refnr": row["refnr"],
            "parse_ok": True,
            "raw_output": raw_output,
            **normalized
        }

    return prediction


rows = [row for _, row in subset.iterrows()]

# Stabiler Baseline-Lauf: nacheinander über ein ggf. geshardetes Modell.
predictions = []

for row in tqdm(rows, total=len(rows)):
    predictions.append(predict_row(row))

print("Fertig. Anzahl Predictions:", len(predictions))
print("Parse-Fails:", sum(not p["parse_ok"] for p in predictions))


100%|██████████| 12/12 [01:12<00:00,  6.05s/it]

Fertig. Anzahl Predictions: 12
Parse-Fails: 0


In [11]:
PRED_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(PRED_PATH, "w", encoding="utf-8") as f:
    for pred in predictions:
        f.write(json.dumps(pred, ensure_ascii=False) + "\n")

print("Gespeichert:", PRED_PATH)


Gespeichert: ../daten/predictions_iterB.jsonl


In [12]:
pred_df = pd.DataFrame(predictions)
pred_df[["refnr", "parse_ok", "homeoffice", "vertragsart", "erfahrungslevel", "gehalt_min_eur", "gehalt_zeitraum", "skills_top3"]]


,refnr,parse_ok,homeoffice,vertragsart,erfahrungslevel,gehalt_min_eur,gehalt_zeitraum,skills_top3
0,15939-BB-633097-7878-9999-S,True,nicht_genannt,festanstellung,mid,NaN,None,"[Python, FMECA, OPUS Suite]"
1,15939-BB-633095-7878-7490-S,True,nicht_genannt,festanstellung,senior,NaN,None,"[OPUS Suite, Life Cycle Costing, Reliability E..."
2,15939-BB-633455-7878-6343-S,True,nicht_genannt,festanstellung,senior,NaN,None,"[Python, Apache Beam, Data Lineage]"
3,15939-BB-633457-7878-2175-S,True,nicht_genannt,festanstellung,senior,NaN,None,"[Machine-Learning, DevOps, Kubernetes]"
4,18777-931781141-S,True,nicht_genannt,festanstellung,mid,NaN,None,"[Siemens Polarion, IBM Rational DOORS, require..."
5,13999-k53401.30280-S,True,teilweise,festanstellung,senior,NaN,None,[]
6,15939-BB-632493-7878-9058-S,True,nicht_genannt,festanstellung,junior,NaN,None,"[Power BI, Excel, Power Point]"
7,13635-7fbe73ac_JB5131141-S,True,remote,festanstellung,egal,NaN,None,"[NLP, Computervision, KI]"
8,20536-lutif851vc-S,True,nicht_genannt,festanstellung,junior,NaN,None,"[AI, Datenerfassung, Publikation]"
9,12826-SA0136034_JB5125696-S,True,nicht_genannt,festanstellung,mid,NaN,None,"[Windows Server, Linux, MS Office 365]"


## Notiz für später: Was wäre Phase 4?

Erst nach der Baseline:
- Prompt verbessern
- Truncation verbessern
- Schema-Regeln im Prompt präzisieren
- Skills anders normalisieren

## Phase 4 — Iteration A

## Phase 4 — Iteration B

## Phase 6 — Voller 7B-Run (gauss)

Die **finale Pipeline** (Iteration-B-Prompt, oben) läuft auf dem **gesamten** `daten/eigener_korpus.jsonl` statt nur auf den 12 Hand-Gold-Anzeigen.

Voraussetzung: alle Zellen oben sind gelaufen (7B-Modell geladen, `build_messages` / `predict_row` definiert). Output: `daten/predictions_7b_full.jsonl` — überschreibt **keine** Baseline-/Iterations-Dateien.

In [ ]:
# ============================================================
# Phase 6 - Voller 7B-Run auf dem gesamten Korpus
# ============================================================
# Nutzt die finale Pipeline (Iteration-B-Prompt) und das oben geladene 7B-Modell.

CORPUS_PATH = Path("../daten/eigener_korpus.jsonl")
FULL_OUT_7B = Path("../daten/predictions_7b_full.jsonl")
FULL_OUT_3B = Path("../daten/predictions_3b_full.jsonl")


def run_full_corpus(out_path):
    """Laesst die aktuelle Pipeline (globales model/tokenizer) ueber den ganzen Korpus laufen."""
    corpus = pd.read_json(CORPUS_PATH, lines=True)
    corpus["refnr"] = corpus["refnr"].astype(str).str.strip()
    print("Korpus-Anzeigen:", len(corpus), "| Modell:", MODEL_NAME)

    preds = []
    for _, row in tqdm(corpus.iterrows(), total=len(corpus)):
        preds.append(predict_row(row))

    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        for p in preds:
            f.write(json.dumps(p, ensure_ascii=False) + "\n")

    print("Gespeichert:", out_path, "| Parse-Fails:", sum(not p["parse_ok"] for p in preds))
    return preds


# 7B ist oben bereits geladen
assert "model" in globals() and "Qwen2.5-7B" in MODEL_NAME, \
    "Erwartet ein geladenes 7B-Modell. Erst die Modell-Lade-Zelle oben mit Qwen2.5-7B ausfuehren."

preds_7b_full = run_full_corpus(FULL_OUT_7B)


## Phase 6 — 3B-Run (euler)

Kontrastlauf mit **`Qwen/Qwen2.5-3B-Instruct`** (`torch_dtype=torch.float32`) auf demselben vollen Korpus und mit **demselben** Iteration-B-Prompt — nur das Modell ändert sich, damit 3B vs. 7B ein fairer Vergleich ist.

Empfohlen: **frischer Spawn auf euler** (Home ist NFS-synchron, Dateien sind da). Die folgende Zelle gibt das 7B-Modell frei, lädt das 3B-Modell und nutzt `run_full_corpus` aus der Zelle oben. Output: `daten/predictions_3b_full.jsonl`.

In [ ]:
# ============================================================
# Phase 6 - 3B-Kontrastlauf auf dem gesamten Korpus
# ============================================================
# Gleicher Prompt, gleiche Pipeline - nur das Modell wechselt auf 3B.
# Voraussetzung: die Zellen oben sind gelaufen (build_messages, predict_row,
# generate_response, run_full_corpus, CORPUS_PATH/FULL_OUT_3B sind definiert).

import gc

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"   # Modellwechsel

# 7B-Modell freigeben, um GPU-Speicher zu schaffen (falls im selben Kernel geladen)
try:
    del model
except NameError:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if torch.cuda.is_available():
    if torch.cuda.device_count() == 1:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float32,
        ).to("cuda:0")
    else:
        max_memory = {i: "14GiB" for i in range(torch.cuda.device_count())}
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float32,
            device_map="auto",
            max_memory=max_memory,
            low_cpu_mem_usage=True,
        )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float32,
    )

model.eval()
# generate_response / predict_row nutzen diese Globals -> nach Reassign passt alles
INPUT_DEVICE = next(model.parameters()).device
print("Modell geladen:", MODEL_NAME, "| Input-Device:", INPUT_DEVICE)

preds_3b_full = run_full_corpus(FULL_OUT_3B)
